In [0]:
from pyspark.sql import functions as F

orders = [
    (101, "Ravi", 1000, "CLOSED"),
    (102, "Sita", 2000, "OPEN"),
    (103, "Ravi", None, "CLOSED"),
    (104, "Arjun", 1500, "CLOSED")
]

df = spark.createDataFrame(
    orders,
    ["order_id", "customer", "amount", "status"]
)

df.show()

+--------+--------+------+------+
|order_id|customer|amount|status|
+--------+--------+------+------+
|     101|    Ravi|  1000|CLOSED|
|     102|    Sita|  2000|  OPEN|
|     103|    Ravi|  NULL|CLOSED|
|     104|   Arjun|  1500|CLOSED|
+--------+--------+------+------+



In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

In [0]:
silver = (
    spark.table("bronze_orders")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("amount").isNotNull())
    .dropDuplicates(["order_id"])
)

silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

In [0]:
gold = (
    spark.table("silver_orders")
    .filter(F.col("status") == "CLOSED")
    .groupBy("customer")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("*").alias("total_orders")
    )
)

gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_revenue")

gold.show()

+--------+-------------+------------+
|customer|total_revenue|total_orders|
+--------+-------------+------------+
|   Arjun|         1500|           1|
|    Ravi|         1000|           1|
+--------+-------------+------------+

